# Dos features para mi proyecto

Proyecto: rebalanceo predictivo de Ecobici (CDMX), flujo de estaciones de agosto de 2026. Archivo: `../../semana-08-sesion-02/data/flujo_estaciones_2026_08.csv`, el mismo de la exploración de la Semana 8.

Aquí pruebas las features antes de decidir cuáles se quedan. Las que se quedaron pasaron a [`pipeline_proyecto.py`](../../semana-09-sesion-02/plantilla-pipeline/pipeline_proyecto.py), en la función `agregar_features`; lo que queda en este notebook es la evidencia de por qué las elegí.

## Paso 1: el momento de la predicción

Proyecto: flujo de estaciones de Ecobici (CDMX), agosto de 2026, 100 estaciones en franjas de 15 minutos (297,600 filas). El modelo predice `y_salidas_prox_30m`: cuántas bicicletas van a salir de una estación en los próximos 30 minutos, y la predicción se hace al cierre de cada franja de 15 minutos, para anticipar si una estación se va a vaciar.

- **Existe en ese instante:** la estación y sus atributos fijos (código, ubicación, `capacidad`), la fecha, la hora, la franja, el día de la semana, si es fin de semana u hora pico, y todo el flujo ya observado: `salidas` y `llegadas` de la franja que acaba de cerrar, `salidas_lag_15m`, `salidas_ultima_hora` y `salidas_mismo_momento_ayer`.
- **Todavía no existe:** las salidas y llegadas de los próximos 30 minutos (el objetivo) y cualquier columna calculada con ellas.

In [1]:
import numpy as np
import pandas as pd

# La misma ruta que usaste en la exploracion de la Semana 8.
# estacion_codigo se lee como texto por los codigos dobles como "271-272".
datos = pd.read_csv(
    "../../semana-08-sesion-02/data/flujo_estaciones_2026_08.csv",
    dtype={"estacion_codigo": str},
)

# La columna que el modelo va a predecir: salidas en los proximos 30 min.
OBJETIVO = "y_salidas_prox_30m"

## Paso 2: feature 1

In [2]:
# Feature 1: periodo de demanda segun la hora. Sale del hallazgo de la
# Semana 8: no hay salidas de 1 a 4 am y la demanda sube 8-9h y 14-18h.
# Solo usa la hora actual, que se conoce al predecir.
hora = datos["hora"]
datos["periodo_demanda"] = np.select(
    [hora < 5, (hora >= 7) & (hora <= 9), (hora >= 14) & (hora <= 18)],
    ["madrugada", "pico_manana", "pico_tarde"],
    default="resto",
)

# Evidencia: el objetivo promedio por periodo.
datos.groupby("periodo_demanda")[OBJETIVO].mean().round(2)

periodo_demanda
madrugada      0.02
pico_manana    2.27
pico_tarde     2.62
resto          1.39
Name: y_salidas_prox_30m, dtype: float64

`periodo_demanda` sale directamente del hallazgo de la exploración de la Semana 8: de 1 a 4 de la mañana prácticamente no hay salidas, y la demanda sube en la mañana (7 a 9) y en la tarde (14 a 18). La evidencia es la diferencia de promedios del objetivo por periodo: madrugada 0.02 salidas en los próximos 30 min, resto 1.39, pico de la mañana 2.27 y pico de la tarde 2.62 (el promedio general es 1.47). Existe al predecir porque solo usa la hora actual. Es categórica sin un orden real: para un modelo lineal (Ridge) usaría one-hot; para un ensamblaje de árboles también funciona con one-hot, y como son solo 4 categorías no agrega muchas columnas.

## Paso 3: feature 2

In [3]:
# Feature 2: salidas de la ultima hora relativas a la capacidad de la
# estacion (demanda reciente por anclaje). Usa solo el pasado y el atributo
# fijo de la estacion. La capacidad minima es 11, asi que no divide entre 0.
datos["salidas_hora_por_capacidad"] = (
    datos["salidas_ultima_hora"] / datos["capacidad"].clip(lower=1)
)

# Evidencia: su correlacion con el objetivo, junto a la de la variable cruda.
datos[["salidas_hora_por_capacidad", "salidas_ultima_hora", OBJETIVO]].corr()[OBJETIVO].round(3)

salidas_hora_por_capacidad    0.541
salidas_ultima_hora           0.600
y_salidas_prox_30m            1.000
Name: y_salidas_prox_30m, dtype: float64

`salidas_hora_por_capacidad` sale del conocimiento del dominio: 5 salidas en una hora no significan lo mismo en una estación de 11 anclajes que en una de 40; dividir entre la capacidad mide qué tan "presionada" está la estación y hace comparables estaciones de distinto tamaño. La evidencia: su correlación con el objetivo es 0.541. Es algo menor que la de `salidas_ultima_hora` sola (0.600), porque el objetivo está en salidas absolutas y no en proporción; su valor es complementar a la variable cruda, no reemplazarla, y sirve sobre todo para el objetivo de alertas de estaciones vacías (que dependen de la capacidad). Existe al predecir porque usa solo la última hora ya observada y un atributo fijo de la estación; la capacidad mínima es 11, así que nunca divide entre 0. Es numérica, no necesita codificación (solo escalado para el modelo lineal).

In [4]:
datos.head()

,estacion_codigo,nombre,latitud,longitud,capacidad,fecha,hora,franja,dia_semana,es_fin_de_semana,es_hora_pico,salidas,llegadas,flujo_neto,salidas_lag_15m,salidas_ultima_hora,salidas_mismo_momento_ayer,y_salidas_prox_30m,periodo_demanda,salidas_hora_por_capacidad
0,007,CE-007 Río Elba - Río Lerma,19.42528,-99.175106,23,2026-08-01,0,0,6,True,False,0,0,0,NaN,NaN,NaN,0.0,madrugada,NaN
1,007,CE-007 Río Elba - Río Lerma,19.42528,-99.175106,23,2026-08-01,0,1,6,True,False,0,0,0,0.0,0.0,NaN,0.0,madrugada,0.0
2,007,CE-007 Río Elba - Río Lerma,19.42528,-99.175106,23,2026-08-01,0,2,6,True,False,0,0,0,0.0,0.0,NaN,0.0,madrugada,0.0
3,007,CE-007 Río Elba - Río Lerma,19.42528,-99.175106,23,2026-08-01,0,3,6,True,False,0,0,0,0.0,0.0,NaN,0.0,madrugada,0.0
4,007,CE-007 Río Elba - Río Lerma,19.42528,-99.175106,23,2026-08-01,1,4,6,True,False,0,0,0,0.0,0.0,NaN,0.0,madrugada,0.0
